# Agentic RAG using Groq API

This notebook implements a simple Retrieval-Augmented Generation (RAG) pipeline using the Groq API for fast language model inference.

The system:
- Stores knowledge in a vector database
- Retrieves relevant context using semantic search
- Generates answers using a Large Language Model (LLM)
- Evaluates the quality of responses

This demonstrates how modern GenAI systems combine retrieval and generation for better accuracy.

## Installation of Required Libraries

In this step, we install all necessary libraries:
- groq: for fast LLM inference
- langchain: for building pipelines
- faiss: for vector similarity search
- sentence-transformers: for embeddings

These tools are essential for building a RAG system.

In [2]:
!pip install -q groq langchain langchain-community faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## Groq API Setup

Here, we initialize the Groq client using an API key.

Groq provides low-latency inference for large language models such as LLaMA, making it suitable for real-time applications.

We define a helper function `groq_llm()` to send prompts to the model and receive responses.

In [ ]:
import os
from groq import Groq

# 🔑 PUT YOUR API KEY HERE
os.environ["GROQ_API_KEY"] = "YOUR_KEY_HERE"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def groq_llm(prompt, max_tokens=300):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

## Knowledge Base and Vector Store

We define a knowledge base containing information related to AI concepts.

Steps performed:
- Split text into smaller chunks
- Convert text into embeddings using Sentence Transformers
- Store embeddings in FAISS for efficient similarity search

This enables fast retrieval of relevant information based on user queries.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

knowledge_text = """
Artificial Intelligence (AI) is a broad field of computer science focused on creating systems capable of performing tasks that normally require human intelligence. These tasks include reasoning, learning from experience, problem-solving, understanding natural language, and perception. AI systems are widely used in applications such as virtual assistants, recommendation systems, fraud detection, and autonomous vehicles.

Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn from data rather than being explicitly programmed. In ML, algorithms are trained on datasets to identify patterns and make predictions. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds hidden patterns in unlabeled data, and reinforcement learning involves learning through trial and error using rewards and penalties.

Deep Learning is a specialized area within machine learning that uses neural networks with multiple layers, known as deep neural networks. These models are particularly effective for tasks such as image recognition, speech recognition, and natural language processing. Deep learning models require large amounts of data and computational power but can achieve very high accuracy.

Natural Language Processing (NLP) is a branch of AI that focuses on enabling computers to understand, interpret, and generate human language. NLP is used in applications such as chatbots, machine translation, sentiment analysis, and voice assistants. Modern NLP systems often use transformer-based architectures like BERT and GPT, which can capture contextual meaning in text more effectively than older models.

BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based model designed for understanding the context of words in a sentence. Unlike traditional models, BERT reads text in both directions (left-to-right and right-to-left), allowing it to better understand meaning. It is commonly used for tasks such as question answering, text classification, and named entity recognition.

Sentence-BERT (SBERT) is an extension of BERT that is optimized for generating sentence embeddings. Instead of processing sentence pairs together, SBERT creates a fixed-size vector for each sentence, enabling fast similarity comparisons using cosine similarity. This makes SBERT ideal for semantic search, clustering, and duplicate detection tasks.

ColBERT (Contextualized Late Interaction over BERT) is a retrieval model that improves search accuracy by performing token-level matching between queries and documents. Instead of compressing an entire sentence into a single vector, ColBERT generates embeddings for each token and computes similarity using a late interaction mechanism. This allows it to capture fine-grained relationships between words and produce more accurate rankings in search systems.

CLIP (Contrastive Language–Image Pretraining) is a multimodal model developed to connect images and text. It is trained using contrastive learning, where the model learns to associate images with their corresponding textual descriptions. CLIP can perform tasks such as zero-shot image classification and image-text retrieval without requiring additional training.

BLIP (Bootstrapping Language-Image Pretraining) is another multimodal model that focuses on both understanding and generating text from images. It combines an image encoder, a multimodal encoder, and a text decoder. BLIP is used for tasks like image captioning and visual question answering, where the model must interpret visual content and produce meaningful text outputs.

Whisper is a speech recognition model designed to convert audio into text. It is trained on large-scale multilingual audio data and can perform transcription and translation across multiple languages. Whisper is robust to noise and accents, making it useful for real-world applications such as meeting transcription and voice assistants.

FAISS (Facebook AI Similarity Search) is a library designed for efficient similarity search and clustering of high-dimensional vectors. It is widely used in retrieval systems, including Retrieval-Augmented Generation (RAG), where it helps quickly find relevant documents based on vector similarity.

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. In a RAG system, relevant documents are first retrieved from a knowledge base, and then a language model generates an answer based on that retrieved context. This approach improves accuracy and reduces hallucination compared to using a language model alone.

Large Language Models (LLMs) are deep learning models trained on massive text datasets to understand and generate human-like language. Examples include GPT, LLaMA, and Mixtral. These models are capable of performing a wide range of tasks such as text generation, summarization, translation, and question answering.

Evaluation of LLMs can be categorized into intrinsic and extrinsic methods. Intrinsic evaluation measures internal model quality using metrics like perplexity, while extrinsic evaluation measures performance on real-world tasks such as question answering or translation. Human evaluation is also commonly used to assess quality, fluency, and usefulness of generated responses.

Agentic AI refers to systems where multiple AI agents collaborate to solve complex problems. These systems often use patterns like prompt chaining, routing, and orchestrator-worker structures to break down tasks and improve performance. Agentic systems are increasingly used in advanced applications like automated research assistants and multi-step reasoning systems.
"""

# Split text
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = [Document(page_content=chunk) for chunk in splitter.split_text(knowledge_text)]

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval-Augmented Generation (RAG)

In this section, we implement the RAG pipeline:

1. Retrieve relevant documents from the vector store
2. Combine them into a context
3. Pass context + question to the LLM
4. Generate an answer grounded in the retrieved data

This reduces hallucination and improves answer accuracy.

In [13]:
def generate_answer(question, context):
    prompt = f"""
Answer ONLY from the context.
If not found, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""
    return groq_llm(prompt)

def retrieve_and_answer(question):
    docs = retriever.invoke(question)
    context = " ".join([d.page_content for d in docs])
    answer = generate_answer(question, context)

    return {
        "question": question,
        "answer": answer,
        "context": context
    }

## Evaluation

We evaluate the generated answers using a simple method:

- Check whether the generated answer is present in the retrieved context

This helps measure whether the model is producing grounded responses.

Although simple, this demonstrates the concept of answer validation.

In [14]:
def evaluate(answer, context):
    # simple check: is answer grounded in context
    score = 1 if answer.lower() in context.lower() else 0
    return score

## Running the Full Pipeline

Here, we:
- Define a set of test questions
- Run them through the RAG pipeline
- Store results including answers and evaluation scores

This simulates real-world usage of the system.

In [15]:
def run_pipeline(question):
    result = retrieve_and_answer(question)

    score = evaluate(result["answer"], result["context"])

    return {
        "question": question,
        "answer": result["answer"],
        "score": score
    }

## Results

The results are displayed in a tabular format using pandas.

This makes it easy to:
- Compare answers
- Check evaluation scores
- Analyze performance

In [16]:
questions = [
    "What is AI?",
    "What is machine learning?",
    "What is NLP?",
    "What is FAISS?",
    "Who is the president of USA?"
]

results = [run_pipeline(q) for q in questions]
results

[{'question': 'What is AI?',
  'answer': 'AI stands for Artificial Intelligence and is a broad field of computer science focused on creating systems capable of performing tasks that normally require human intelligence.',
  'score': 0},
 {'question': 'What is machine learning?',
  'answer': 'Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data rather than being explicitly programmed.',
  'score': 0},
 {'question': 'What is NLP?',
  'answer': 'A branch of AI that focuses on enabling computers to understand, interpret, and generate human language.',
  'score': 1},
 {'question': 'What is FAISS?',
  'answer': 'A library designed for efficient similarity search and clustering of high-dimensional vectors.',
  'score': 1},
 {'question': 'Who is the president of USA?',
  'answer': "I don't know.",
  'score': 0}]

In [17]:
import pandas as pd
pd.DataFrame(results)

,question,answer,score
0,What is AI?,AI stands for Artificial Intelligence and is a...,0
1,What is machine learning?,Machine Learning is a subset of Artificial Int...,0
2,What is NLP?,A branch of AI that focuses on enabling comput...,1
3,What is FAISS?,A library designed for efficient similarity se...,1
4,Who is the president of USA?,I don't know.,0


## Reflection

In this lab, I implemented a Retrieval-Augmented Generation (RAG) system using the Groq API for fast inference.

I learned how to combine vector databases (FAISS) with language models to improve the accuracy of generated responses. Instead of relying only on the model’s internal knowledge, the system retrieves relevant information and uses it to generate grounded answers.

One key insight was that retrieval significantly reduces hallucination and improves reliability. I also understood the importance of embeddings and similarity search in modern AI systems.

Using Groq API helped achieve low-latency responses, making the system more practical for real-time applications.

Overall, this lab helped me understand how real-world GenAI systems are built using a combination of retrieval, generation, and evaluation techniques.